# 04z_fantrax_crosswalk

**Purpose:** Resolve Fantrax `scorer_id` to the player registries so
`fact_fantrax_adp` rows carry real foreign keys.

- Primary key: `gsis_id` via `dim_nfl_players` — the full nflverse registry already
  holds all veterans + signed rookies, so it covers ~100% of the board.
- Secondary: `player_key` via `dim_rookie_prospect` for current draft-class rookies.

**Matching:** manual overrides (`data/raw/fantrax_crosswalk_overrides.csv`:
`scorer_id -> gsis_id` or `"new"`) -> exact cleaned-name -> disambiguate by position / active status /
recency -> fuzzy fallback (`token_sort_ratio`: auto >=90, review 70-89, unmatched <70).

**Outputs:**
- `data/dim_fantrax_crosswalk.parquet` — scorer_id -> gsis_id / player_key (+ method, score)
- `review_fantrax_crosswalk.csv` — rows needing manual review (only when any)
- back-fills `gsis_id` / `player_key` into `data/fact_fantrax_adp.parquet`


In [1]:
# ---- Setup & Config (shared module) ----------------------------------------
# All config + path anchoring + helpers live in notebooks/etl_helpers.py.
# CFG.data_dir / DATA / REVIEW are anchored to the repo root -> writes always
# land in <repo>/data no matter the CWD this notebook runs from.
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import (LeagueConfig, CFG, DATA, REVIEW, TODAY, ALIAS,
                         clean_name_for_match, generate_player_key, parse_height_to_inches)

# ---- Setup & Config ---------------------------------------------------------
from dataclasses import dataclass
from datetime import date
from pathlib import Path
import glob
import json
import re

import pandas as pd
from thefuzz import fuzz, process

In [2]:
# ---- Helpers ----------------------------------------------------------------
# Name normalization for matching is shared: etl_helpers.clean_name_for_match
# (strips periods / apostrophes / generational suffixes, lowercases). Same rule
# used by resolve_dynasty_crosswalk (04b/04x) so Fantrax and dynasty sources
# match the registry identically.


def pos_tokens(raw: str) -> set:
    """Fantrax position string -> set of upper codes ('WR,DB' -> {WR, DB})."""
    return {p.strip().upper() for p in str(raw).split(",") if p.strip()}

In [3]:
# ---- Load -------------------------------------------------------------------
fact = pd.read_parquet(CFG.path(CFG.fact_name))
# one row per scorer_id (latest snapshot) — crosswalk is identity, not time series
fact_latest = (fact.sort_values(["season", "week"])
                    .drop_duplicates("scorer_id", keep="last"))

npl = pd.read_parquet(CFG.path(CFG.nfl_players_name)).copy()
rp = pd.read_parquet(CFG.path(CFG.rookie_prospect_name)).copy()

npl["name_clean"] = npl["display_name"].map(clean_name_for_match)
rp["name_clean"] = rp["player_name"].map(clean_name_for_match)

# rookie-prospect lookup: cleaned name -> player_key (first if dup)
rp_lookup = (rp.dropna(subset=["name_clean"])
               .drop_duplicates("name_clean")
               .set_index("name_clean")["player_key"].to_dict())

# manual override store: scorer_id -> gsis_id (or "new" = not in the registry yet).
# Persistent across full re-runs - add decisions to the CSV, never inline here.
OVERRIDES_CSV = DATA / "raw" / "fantrax_crosswalk_overrides.csv"
overrides = (pd.read_csv(OVERRIDES_CSV, dtype=str).set_index("scorer_id")["gsis_id"].to_dict()
             if OVERRIDES_CSV.exists() else {})

# ---- Extend universe: scorer_ids from the live draft board, not just ADP ---
# fact_fantrax_adp only covers Fantrax's ranked/ADP-tracked pool. Draft picks
# (deep bench, IDP, un-ranked rookies) can carry scorer_ids that never show up
# there, so they'd silently fall through to null gsis_id/player_key downstream
# (dim_roster_asset, 02d) without ever being flagged in the review CSV. Union
# them into fact_latest here so they run through the same match_one pipeline.
draft_scorers = {}
for _f in sorted(glob.glob(str(DATA / "raw" / "fantrax_draftresults_2026*.json"))):
    _raw = json.load(open(_f, encoding="utf-8"))
    for s in _raw["responses"][0]["data"].get("scorers", []):
        draft_scorers.setdefault(s["scorerId"], s)   # first file wins on dup

_known = set(fact_latest["scorer_id"])
_extra = pd.DataFrame([
    {
        "scorer_id": sid,
        "player_name": s["name"],
        "position_raw": re.sub(r"<[^>]+>", "", s.get("posShortNames", "")),
        "nfl_team": s.get("teamShortName", ""),
        "is_rookie": bool(s.get("rookie", False)),
    }
    for sid, s in draft_scorers.items() if sid not in _known
])
if len(_extra):
    fact_latest = pd.concat([fact_latest, _extra], ignore_index=True)
print(f"[info] +{len(_extra)} draft-only scorer_ids added to match universe "
      f"(not in fact_fantrax_adp)")

print(f"fantrax players: {len(fact_latest)} | dim_nfl_players: {len(npl)} | "
      f"dim_rookie_prospect: {len(rp)} | manual overrides: {len(overrides)}")

[info] +78 draft-only scorer_ids added to match universe (not in fact_fantrax_adp)
fantrax players: 1734 | dim_nfl_players: 25036 | dim_rookie_prospect: 468 | manual overrides: 33


In [4]:
# ---- Match: scorer_id -> gsis_id (+ player_key) -----------------------------
# Index dim_nfl_players candidates by cleaned name for O(1) lookup.
npl_by_name = {n: g for n, g in npl.groupby("name_clean")}
npl_clean_names = list(npl_by_name.keys())


def disambiguate(cands: pd.DataFrame, fpos: set) -> pd.Series:
    """Pick the right candidate among same-name players: position, then active, then recency."""
    df = cands
    if fpos:
        m = (df["position"].str.upper().isin(fpos)
             | df["position_group"].str.upper().isin(fpos))
        if m.any():
            df = df[m]
    if (df["status"] == "ACT").any():
        df = df[df["status"] == "ACT"]
    if "entry_year" in df and df["entry_year"].notna().any():
        df = df.sort_values("entry_year", ascending=False)
    return df.iloc[0]


def match_one(row) -> dict:
    cn = clean_name_for_match(row["player_name"])
    fpos = pos_tokens(row["position_raw"])
    method, score, gsis = "unmatched", 0, None

    ov = overrides.get(row["scorer_id"])
    if ov == "new":
        method = "new"  # confirmed absent from the registry - skip review staging
    elif ov is not None:
        gsis, method, score = ov, "manual", 100
    elif cn in npl_by_name:
        cands = npl_by_name[cn]
        pick = cands.iloc[0] if len(cands) == 1 else disambiguate(cands, fpos)
        gsis = pick["gsis_id"]
        method = "exact" if len(cands) == 1 else "exact+disambig"
        score = 100
    else:
        best_name, best = process.extractOne(
            cn, npl_clean_names, scorer=fuzz.token_sort_ratio
        )
        score = best
        if best >= CFG.fuzzy_auto_threshold:
            pick = disambiguate(npl_by_name[best_name], fpos)
            gsis, method = pick["gsis_id"], "fuzzy"
        elif best >= CFG.fuzzy_review_threshold:
            method = "review"
        else:
            method = "unmatched"

    return {
        "scorer_id": row["scorer_id"],
        "player_name": row["player_name"],
        "position_raw": row["position_raw"],
        "nfl_team": row["nfl_team"],
        "is_rookie": row["is_rookie"],
        "gsis_id": gsis,
        "player_key": rp_lookup.get(cn),
        "match_method": method,
        "match_score": int(score),
    }


xwalk = pd.DataFrame([match_one(r) for _, r in fact_latest.iterrows()])
print(xwalk["match_method"].value_counts().to_string())

match_method
exact             1580
exact+disambig     113
manual              26
fuzzy                8
new                  7


In [5]:
# ---- Persist crosswalk + review staging -------------------------------------
xwalk["resolved_date"] = date.today().isoformat()
xwalk.to_parquet(CFG.path(CFG.crosswalk_name), index=False)

review = xwalk[xwalk["match_method"].isin(["review", "unmatched"])]
review_path = str(REVIEW / "review_fantrax_crosswalk.csv")
if len(review):
    out = review.copy()
    out["action"] = ""          # user fills: a gsis_id value, or "new"
    out.to_csv(review_path, index=False)
    print(f"[review] {len(review)} rows need manual review -> {review_path}")
else:
    print("[review] none - all rows auto-resolved")
    # repo convention: a fully-filled review file is applied work - archive it
    # as *.applied_YYYYMMDD.csv so a bare review_*.csv always means pending.
    stale = Path(review_path)
    if stale.exists():
        old = pd.read_csv(stale)
        acts = old["action"].fillna("").astype(str).str.strip() if "action" in old else None
        if acts is not None and len(acts) and acts.ne("").all():
            try:
                stale.rename(stale.with_suffix(f".applied_{date.today():%Y%m%d}.csv"))
                print(f"[review] archived filled review -> "
                      f"{stale.stem}.applied_{date.today():%Y%m%d}.csv")
            except OSError as e:
                print(f"[warn] could not archive filled review (file open in Excel?): {e}")

print(f"[ok] wrote {len(xwalk)} crosswalk rows -> {CFG.path(CFG.crosswalk_name)}")


[review] none - all rows auto-resolved
[ok] wrote 1734 crosswalk rows -> C:\Users\benha\OneDrive\Documents\GitHub\Python-PowerBI-DynastyFantasyFootball\data/dim_fantrax_crosswalk.parquet


In [6]:
# ---- Back-fill FKs into fact_fantrax_adp ------------------------------------
fact = pd.read_parquet(CFG.path(CFG.fact_name))
key = xwalk.set_index("scorer_id")[["gsis_id", "player_key"]]
fact["gsis_id"] = fact["scorer_id"].map(key["gsis_id"])
fact["player_key"] = fact["scorer_id"].map(key["player_key"])
fact.to_parquet(CFG.path(CFG.fact_name), index=False)

print(f"[ok] back-filled fact_fantrax_adp — {fact['gsis_id'].notna().sum()}/"
      f"{len(fact)} rows have gsis_id "
      f"({fact['gsis_id'].notna().mean() * 100:.1f}%)")


[ok] back-filled fact_fantrax_adp — 1649/1656 rows have gsis_id (99.6%)


In [7]:
# ---- Validation report ------------------------------------------------------
print("=== crosswalk coverage ===")
print(f"total scorer_ids:       {len(xwalk)}")
print(f"gsis_id resolved:       {xwalk['gsis_id'].notna().sum()}")
print(f"player_key (rookies):   {xwalk['player_key'].notna().sum()}")
print(f"needs review/unmatched: {xwalk['match_method'].isin(['review', 'unmatched']).sum()}")
print()
print("by method:")
print(xwalk["match_method"].value_counts().to_string())
print()
# sanity: any gsis_id claimed by >1 scorer_id? Usually two same-name players
# (e.g. the two Jaylon Joneses) that both auto-resolved to one id. This is real
# and recurring, so don't hard-stop the whole pipeline on it — log it, stage the
# colliding rows to a review CSV for a human to disambiguate via the overrides
# file, and continue. The crosswalk is still written; the collision is flagged,
# not fatal. (Was a RuntimeError until 2026-06-13.)
dups = xwalk[xwalk["gsis_id"].notna()].groupby("gsis_id").size()
dups = dups[dups > 1]
if len(dups):
    collisions = (xwalk[xwalk["gsis_id"].isin(dups.index)]
                  [["scorer_id", "player_name", "position_raw", "nfl_team",
                    "gsis_id", "match_method", "match_score"]]
                  .sort_values(["gsis_id", "player_name"]))
    collision_path = str(REVIEW / "review_fantrax_crosswalk_collisions.csv")
    out = collisions.copy()
    out["action"] = ""   # user fills the correct gsis_id (or "new") per scorer_id
    out.to_csv(collision_path, index=False)
    print(f"[warn] {len(dups)} gsis_id claimed by >1 scorer_id "
          f"({len(collisions)} rows) -> {collision_path}")
    print(f"[warn] resolve in {OVERRIDES_CSV} (same-name players need explicit "
          f"ids), then re-run. Continuing with the collision left in place.")
    print(collisions.to_string(index=False))
else:
    print("[ok] gsis_id mapping is 1:1")


=== crosswalk coverage ===
total scorer_ids:       1734
gsis_id resolved:       1727
player_key (rookies):   266
needs review/unmatched: 0

by method:
match_method
exact             1580
exact+disambig     113
manual              26
fuzzy                8
new                  7

[ok] gsis_id mapping is 1:1
